# Ejercicios Spark DataFrames 

Vamos a practicar un poco con tus nuevas habilidades de Spark DataFrame, se te harán algunas preguntas básicas sobre algunos datos del mercado de valores, en este caso Walmart Stock de los años 2012-2017. 

Responde a las preguntas y completa las tareas de abajo.

#### ¡Utiliza el archivo walmart_stock.csv para responder y completar las tareas siguientes!

#### Iniciar una sesión de Spark

#### Preparar entorno y arrancar Spark

Esta celda esta pensada para ejecutarse de principio a fin en local o en Google Colab. Si PySpark ya esta instalado, no lo reinstala; si falta y estas en Colab, instala Java 17 y PySpark antes de crear la sesion.


In [ ]:
import importlib.util
import os
import subprocess
import sys
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules or "COLAB_RELEASE_TAG" in os.environ
JAVA_HOME = "/usr/lib/jvm/java-17-openjdk-amd64"
PYSPARK_VERSION = "4.0.1"


def run_command(command):
    print("Ejecutando:", " ".join(command))
    subprocess.run(command, check=True)


def java_available():
    if Path(JAVA_HOME).exists():
        return True
    return subprocess.run(["which", "java"], capture_output=True).returncode == 0


if IN_COLAB:
    if not java_available():
        print("Java 17 no esta disponible. Instalando Java 17 para Colab...")
        run_command(["apt-get", "update", "-qq"])
        run_command(["apt-get", "install", "-y", "openjdk-17-jdk-headless", "-qq"])
    else:
        print("Java disponible. Se reutiliza el entorno actual.")

    os.environ["JAVA_HOME"] = JAVA_HOME
    os.environ["PATH"] += f":{JAVA_HOME}/bin"

    if importlib.util.find_spec("pyspark") is None:
        print("PySpark no esta instalado. Instalando PySpark para Colab...")
        run_command([sys.executable, "-m", "pip", "install", "-q", "--force-reinstall", f"pyspark=={PYSPARK_VERSION}"])
    else:
        print("PySpark ya esta instalado. Se reutiliza el entorno actual.")
else:
    if importlib.util.find_spec("pyspark") is None:
        raise ModuleNotFoundError(
            "PySpark no esta instalado. En local instala las dependencias con: "
            "python -m pip install -r requirements.txt"
        )
    print("Entorno local con PySpark disponible.")

from pyspark.sql import SparkSession

spark = (SparkSession.builder
         .master("local[*]")
         .appName("Practica Spark DataFrame - Walmart Stock")
         .getOrCreate())

spark.sparkContext.setLogLevel("WARN")
spark.range(10).show()
print("Spark listo")

#### Cargar el archivo CSV de Walmart Stock, hacer que Spark infiera los tipos de datos.

In [ ]:
df = (spark.read
      .option("header", True)
      .option("inferSchema", True)
      .csv("walmart_stock.csv"))

df.show(5)

#### ¿Cuáles son los nombres de las columnas?

In [ ]:
df.columns

#### ¿Qué aspecto tiene el esquema?

In [ ]:
df.printSchema()

#### Imprime las 5 primeras columnas.

In [ ]:
for row in df.head(5):
    print(row)
    print()

#### Utiliza describe() para conocer el DataFrame.

In [ ]:
df.describe().show()

#### Hay demasiados decimales para la media y el stddev en describe(). Formatea los números para que sólo se muestren hasta dos decimales. Presta atención a los tipos de datos que devuelve .describe()

In [ ]:
summary = df.describe()
summary.printSchema()

In [ ]:
from pyspark.sql.functions import format_number

formatted_summary = summary.select(
    summary["summary"],
    format_number(summary["Open"].cast("double"), 2).alias("Open"),
    format_number(summary["High"].cast("double"), 2).alias("High"),
    format_number(summary["Low"].cast("double"), 2).alias("Low"),
    format_number(summary["Close"].cast("double"), 2).alias("Close"),
    summary["Volume"].cast("int").alias("Volume")
)

In [ ]:
formatted_summary.show()

#### Crea un nuevo dataframe con una columna llamada HV Ratio que es la relación entre el precio máximo y el volumen de las acciones negociadas durante un día.

In [ ]:
df_hv = df.withColumn("HV Ratio", df["High"] / df["Volume"])
df_hv.select("HV Ratio").show()

#### ¿Qué día hubo el pico máximo en el precio?

In [ ]:
df.orderBy(df["High"].desc()).select("Date", "High").show(1)

# Si se quiere solo el valor de la fecha:
df.orderBy(df["High"].desc()).head()["Date"]

#### ¿Cuál es la media de la columna Close?

In [ ]:
from pyspark.sql.functions import avg

df.select(avg("Close").alias("Media Close")).show()

#### ¿Cuál es el máximo y el mínimo de la columna Volumen?

In [ ]:
from pyspark.sql.functions import max, min

In [ ]:
df.select(
    max("Volume").alias("Max Volume"),
    min("Volume").alias("Min Volume")
).show()

#### ¿Cuántos días estuvo el cierre por debajo de los 60 dólares?

In [ ]:
df.filter(df["Close"] < 60).count()

#### ¿Qué porcentaje de veces el Máximo fue superior a 80 dólares?
#### En otras palabras, (Número de días de máximos>80)/(Días totales en el conjunto de datos)

In [ ]:
dias_high_mayor_80 = df.filter(df["High"] > 80).count()
total_dias = df.count()
(dias_high_mayor_80 / total_dias) * 100

#### ¿Cuál es la correlación de Pearson entre High y Volume?

In [ ]:
from pyspark.sql.functions import corr

df.select(corr("High", "Volume").alias("Correlacion High-Volume")).show()

#### ¿Cuál es el valor máximo de High por año?

In [ ]:
from pyspark.sql.functions import year

df_year = df.withColumn("Year", year(df["Date"]))
df_year.groupBy("Year").max("High").orderBy("Year").show()

#### ¿Cuál es el cierre medio de cada mes del calendario?
#### En otras palabras, a lo largo de todos los años, ¿cuál es el precio medio de cierre para enero, febrero, marzo, etc.? Su resultado tendrá un valor para cada uno de estos meses. 

In [ ]:
from pyspark.sql.functions import month

df_month = df.withColumn("Month", month(df["Date"]))
df_month.groupBy("Month").avg("Close").orderBy("Month").show()